# Phase 5: Model Training & Benchmarking
Train and evaluate multiple models for 15-minute AQ prediction.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

# Load data
X_train = np.load('X_train_15T.npy')
X_test = np.load('X_test_15T.npy')
y_train = pd.read_csv('y_train_15T.csv', index_col=0).values.ravel()
y_test = pd.read_csv('y_test_15T.csv', index_col=0).values.ravel()
scaler = joblib.load('scaler_15T.pkl')
feature_names = joblib.load('feature_names_15T.pkl')

print(f"Data loaded: Train {X_train.shape}, Test {X_test.shape}")

# Initialize results tracker
results = []

def evaluate_model(model_name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'model_name': model_name, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

# Baseline: Persistence Model
y_persist = X_test[:, feature_names.index('pm2_5_lag_15m')]  # t-15m as prediction
persist_result = evaluate_model('Persistence', y_test, y_persist)
results.append(persist_result)
print(f"Persistence: RMSE={persist_result['RMSE']:.3f}, MAE={persist_result['MAE']:.3f}, R2={persist_result['R2']:.3f}")

# Save results
pd.DataFrame(results).to_csv('results.csv', index=False)
print("Results saved to results.csv")

In [ ]:
# XGBoost
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.1, 0.2]
}

xgb = XGBRegressor(random_state=42)
tscv = TimeSeriesSplit(n_splits=5)
grid_search = GridSearchCV(xgb, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_xgb = grid_search.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)
xgb_result = evaluate_model('XGBoost', y_test, y_pred_xgb)
results.append(xgb_result)

# Save model
joblib.dump(best_xgb, 'best_xgboost.pkl')
print(f"XGBoost: RMSE={xgb_result['RMSE']:.3f}, MAE={xgb_result['MAE']:.3f}, R2={xgb_result['R2']:.3f}")

# Update results
pd.DataFrame(results).to_csv('results.csv', index=False)

In [ ]:
# Random Forest
from sklearn.ensemble import RandomForestRegressor

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5]
}

rf = RandomForestRegressor(random_state=42)
grid_search_rf = GridSearchCV(rf, param_grid_rf, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
grid_search_rf.fit(X_train, y_train)

best_rf = grid_search_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)
rf_result = evaluate_model('RandomForest', y_test, y_pred_rf)
results.append(rf_result)

# Save model
joblib.dump(best_rf, 'best_rf.pkl')
print(f"RandomForest: RMSE={rf_result['RMSE']:.3f}, MAE={rf_result['MAE']:.3f}, R2={rf_result['R2']:.3f}")

# Update results
pd.DataFrame(results).to_csv('results.csv', index=False)